# Orientation distributions

In [ ]:
from apsg import *

## Distribution objects
**APSG** provides probability distributions of orientation data on the circle (2D directions) and on the sphere (lines, poles and vectors). *Vectorial* models distinguish a vector from its antipode (`dist_fisher`, `dist_kent`, `dist_esag`, `dist_vonmises`, `dist_wrapped_normal`), while *axial* models treat a line and its antipode as the same orientation, as needed for lineations and foliations (`dist_watson`, `dist_bingham`, `dist_acg`, `dist_axial_vonmises`).

A distribution is created from its parameters. Kent distribution has the mean direction, the major axis of its elliptical contours, the concentration `kappa` and the ellipticity `beta`:

In [ ]:
d = dist_kent(vec(120, 40), vec(30, 0), kappa=40, beta=15)
d

All parameters are available as dictionary using `params` property

In [ ]:
d.params

The `pdf` method evaluates the density (with respect to area on the sphere, so the uniform density is 1/4π) for a feature, a feature set or an array of vectors

In [ ]:
d.pdf(vec(120, 40))

Random samples are generated by `rvs` method. The `rng` argument (an integer seed or a `numpy.random.Generator`) makes the sample reproducible and `feature` selects the feature class of the returned set (`Vector3Set` by default for vectorial models).

In [ ]:
v = d.rvs(150, rng=7)
s = StereoNet()
s.vector(v, ms=4)
s.show()

## Fitting
Any feature set could be fitted by `fit` method, where the model is given by its short name (`fisher`, `kent`, `esag`, `watson`, `bingham`, `acg`, `vonmises`, `axial_vonmises`, `wrapped_normal` or `lisle`). The default method is maximum likelihood.

In [ ]:
k = v.fit('kent')   # same as dist_kent.fit(v)
k

The `fit_info` property holds the sample size, the log-likelihood and whether the solver converged

In [ ]:
k.fit_info

The `mean_confidence` method returns the confidence region of the mean direction, a cone (`alpha`) or an ellipse (`axes` and `gamma` semi-angles in degrees)

In [ ]:
k.mean_confidence(level=0.95)

## Model selection
Fitted models could be compared by Akaike (`aic`) or Bayesian (`bic`) information criterion, lower is better. The elliptical Kent and ESAG distributions describe our sample much better than the circular Fisher distribution:

In [ ]:
for model in ['fisher', 'kent', 'esag']:
    print(f'{model:7s} BIC: {v.fit(model).bic(v):8.2f}')

Note that only models of the same symmetry should be compared this way. An axial density spreads its probability over both antipodes, so it can't be compared with vectorial one.

## Axial data
Lineations and foliations are axial data, so the appropriate models are Watson distribution with circular contours and Bingham distribution with elliptical contours. The Watson distribution is bipolar (clustered) for positive `kappa`:

In [ ]:
l = dist_watson(lin(210, 35), 15).rvs(100, rng=1, feature=lin)
l.fit('watson')

and forms a girdle for negative `kappa`, with the mean axis as the pole of the girdle. A typical example are poles of folded foliations, where the pole of the girdle is the fold axis:

In [ ]:
f = dist_watson(lin(120, 30), -20).rvs(100, rng=2, feature=fol)
f.fit('watson')

Bingham distribution covers clusters, girdles and everything between. For a girdle `kappa_major` is near 0, and the minor axis is the pole of the girdle. The mean, major and minor axes are available as `axes` property:

In [ ]:
b = f.fit('bingham')
b

In [ ]:
b.axes

In [ ]:
s = StereoNet()
s.point(f, ms=4)
s.point(f.fit('watson').mean, color='r', marker='*', ms=12)  # fold axis
s.show()

Fitting a model of vectorial data to axial data (e.g. `fisher` to a `LineationSet`) issues a warning and fits the data flipped into one hemisphere (see `halfspace` method), which is adequate only for clustered data.

In [ ]:
l.fit('fisher')

## Plotting densities and confidence regions
`StereoNet` provides `density` method to contour the density of a distribution in multiples of uniform distribution. The `confidence` method with `method='kent'` or `method='esag'` draws the confidence ellipse of the mean of the fitted model, which could be compared with the circular Fisher's cone:

In [ ]:
v = dist_kent(vec(120, 40), vec(30, 0), kappa=12, beta=5).rvs(30, rng=3)
s = StereoNet()
s.density(v.fit('kent'), levels=5)
s.vector(v, color='k', ms=4)
s.confidence(v, method='kent', color='r', label='Kent')
s.confidence(v, method='fisher', color='b', label='Fisher')
s.show()

In [ ]:
s = StereoNet()
s.density(b, levels=5, cmap='Greens')
s.point(f, color='k', ms=3)
s.show()

## Circular data
For 2D data, `dist_vonmises` and `dist_wrapped_normal` describe directions with sense (`Vector2Set`) and `dist_axial_vonmises` describes axial directions like strikes or trends of lines (`Direction2Set`).

In [ ]:
c = dist_vonmises(60, kappa=4).rvs(100, rng=3)
c.fit('vonmises')

In [ ]:
dd = dist_axial_vonmises(20, kappa=3).rvs(100, rng=4, feature=dir2)
ad = dd.fit('axial_vonmises')
ad

`RosePlot` provides `density` method to plot circular distribution in the same units as `pdf` method

In [ ]:
r = RosePlot()
r.pdf(dd)
r.density(ad, fc='none', ec='r')
r.show()

## Pairs and faults
Paired data (foliation and lineation) are described by Lisle distribution (`dist_lisle`). `kappa1` measures the concentration of the lineation within the foliation and `kappa2` the concentration of the foliation about the lineation. It does not depend on the sense of lineation or foliation normal, so it could be used for faults as well.

In [ ]:
p = dist_lisle(pair(120, 40, 160, 34), kappa1=4, kappa2=8).rvs(30, rng=5)
p.fit('lisle')

The `lisle_statistics` method of `PairSet` returns the mean pair after Lisle (1989), together with confidence ellipses of the mean lineation and foliation normal, which are plotted by `confidence` method of `StereoNet`:

In [ ]:
stats = p.lisle_statistics()
stats['mean']

In [ ]:
s = StereoNet()
s.pair(p)
s.pair(stats['mean'], color='r')
s.confidence(p, color='r')
s.show()

## Statistics methods of feature sets
The statistics methods of feature sets are built on these distributions, e.g. `watson_statistics` and `bingham_statistics` use fitted Watson and Bingham distributions and `random_fisher`, `random_kent` or `random_vonmises` sample the corresponding models. The resultant based statistics of axial data (`fisher_statistics`, `var`, `delta`, `rdegree` and `csd` of lineation and foliation sets) are deprecated, use Watson or Bingham statistics or distributions instead.

In [ ]:
l.watson_statistics()